# Hybrid Retrieval Evaluation — Ablation Study

**Người 3 (Kiệt)** — Sparse retrieval, Hybrid, RRF, Cross-Encoder reranker ablation.

This notebook runs the full ablation for 5 configs:

| # | Config | Dense | Sparse | RRF | Cross-Encoder | Mục tiêu |
|---|--------|-------|--------|-----|---------------|---------|
| 1 | `Retrieval-Hybrid-SparseDense` | ✅ | ✅ | ❌ | ❌ | So sánh Hybrid với Dense |
| 2 | `Rerank-None-Hybrid` | ✅ | ✅ | ❌ | ❌ | Hybrid không rerank (baseline) |
| 3 | `Rerank-RRF-Hybrid` | ✅ | ✅ | ✅ | ❌ | Hybrid dùng RRF |
| 4 | `Rerank-CrossEncoder-Hybrid` | ✅ | ✅ | ❌ | ✅ | Cross-Encoder only |
| 5 | `Rerank-RRFPlusCrossEncoder-Hybrid` | ✅ | ✅ | ✅ | ✅ | RRF + CE rerank |

**No rerank** = simple score-based merge + dedup (không RRF, không CE).

**Pipeline**: `query → Dense search → Sparse search → Fusion → (optional rerank) → metrics`

---

## 1. Setup & Install Dependencies

In [ ]:
# === INSTALL DEPENDENCIES ===
# Uncomment the line below when running on Kaggle/Colab
# !pip install -q rank_bm25 sentence-transformers faiss-cpu underthesea

import json
import os
import sys
import time
import math
import pickle
import hashlib
import re
import unicodedata
from pathlib import Path
from dataclasses import dataclass, field
from typing import Any
from collections import defaultdict
from datetime import datetime, timezone

import numpy as np

print('Python version:', sys.version)
print('Setup complete.')

## 2. Configuration

In [ ]:
# === PATHS — Edit these for your environment ===
# On Kaggle: mount your dataset and set paths accordingly
# On local: paths relative to project root

# Try to detect environment
IS_KAGGLE = os.path.exists('/kaggle/input')
IS_COLAB = 'google.colab' in sys.modules

if IS_KAGGLE:
    # Kaggle paths — adjust dataset name as needed
    PROJECT_ROOT = Path('/kaggle/working')
    DATA_DIR = Path('/kaggle/input/textmining-data')  # <-- adjust to your dataset name
    INDEX_DIR = DATA_DIR / 'faiss_index'
    CHUNKS_PATH = DATA_DIR / 'v2' / 'chunks.jsonl'
    BENCHMARK_PATH = DATA_DIR / 'benchmark' / 'qa_final.jsonl'
    SPARSE_INDEX_DIR = PROJECT_ROOT / 'sparse_index'
    OUTPUT_BASE = PROJECT_ROOT / 'evaluation_runs' / 'ablation'
elif IS_COLAB:
    # Colab paths
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/TextMining')
    DATA_DIR = PROJECT_ROOT / 'data'
    INDEX_DIR = DATA_DIR / 'faiss_index'
    CHUNKS_PATH = DATA_DIR / 'v2' / 'chunks.jsonl'
    BENCHMARK_PATH = DATA_DIR / 'benchmark' / 'qa_final.jsonl'
    SPARSE_INDEX_DIR = DATA_DIR / 'sparse_index'
    OUTPUT_BASE = PROJECT_ROOT / 'evaluation_runs' / 'ablation'
else:
    # Local development
    PROJECT_ROOT = Path('.').resolve()
    if (PROJECT_ROOT / 'src').exists():
        pass  # already at project root
    elif (PROJECT_ROOT.parent / 'src').exists():
        PROJECT_ROOT = PROJECT_ROOT.parent
    DATA_DIR = PROJECT_ROOT / 'data'
    INDEX_DIR = DATA_DIR / 'faiss_index'
    # Auto-detect chunks path
    for _p in [DATA_DIR / 'v2' / 'chunks.jsonl', DATA_DIR / 'pre-processed' / 'chunks.jsonl']:
        if _p.exists():
            CHUNKS_PATH = _p
            break
    else:
        CHUNKS_PATH = DATA_DIR / 'v2' / 'chunks.jsonl'
    BENCHMARK_PATH = DATA_DIR / 'benchmark' / 'qa_final.jsonl'
    SPARSE_INDEX_DIR = DATA_DIR / 'sparse_index'
    OUTPUT_BASE = PROJECT_ROOT / 'evaluation_runs' / 'ablation'

# Add src to path for imports
SRC_DIR = PROJECT_ROOT / 'src'
if SRC_DIR.exists() and str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# === RETRIEVAL SETTINGS ===
EMBEDDING_MODEL = 'intfloat/multilingual-e5-large'
CROSS_ENCODER_MODEL = 'cross-encoder/mmarco-mMiniLMv2-L12-H384-v1'
TOP_K = 30          # candidates from each retriever
TOP_N = 10          # final results after fusion/rerank
SCORE_THRESHOLD = 0.30
RRF_K = 60          # RRF smoothing constant
FILTER_PROFILE = 'broad'
TOP_K_EVAL = [1, 3, 5, 10]  # k values for Recall@k, MRR@k, etc.

# === ABLATION LIMIT (set None for full run) ===
ABLATION_LIMIT = None  # e.g. 10 for smoke test, None for full

print('Environment:', 'Kaggle' if IS_KAGGLE else 'Colab' if IS_COLAB else 'Local')
print('PROJECT_ROOT:', PROJECT_ROOT)
print('DATA_DIR:', DATA_DIR)
print('INDEX_DIR:', INDEX_DIR)
print('CHUNKS_PATH:', CHUNKS_PATH)
print('BENCHMARK_PATH:', BENCHMARK_PATH)
print('SPARSE_INDEX_DIR:', SPARSE_INDEX_DIR)
print('OUTPUT_BASE:', OUTPUT_BASE)

## 3. Core Classes (Inline for Kaggle portability)

These classes are copies of `src/retrieval/sparse_retriever.py` and `src/retrieval/hybrid_retriever.py` — inlined here so the notebook is self-contained on Kaggle.

In [ ]:
# === SHARED SCHEMA ===

@dataclass(frozen=True)
class SearchHit:
    point_id: str
    score: float
    payload: dict[str, Any]

@dataclass(frozen=True)
class RetrievedChunk:
    chunk_id: str
    chunk_text: str
    citation_anchor: str
    citation_label: str
    title: str
    article_number: str | None
    unit_type: str
    path: str | None
    validity_group: str
    legal_authority_rank: int
    vector_score: float
    rerank_score: float
    id_str: str
    parent_unit_id: str
    metadata: dict[str, Any] = field(default_factory=dict)

@dataclass(frozen=True)
class RetrievalResult:
    chunks: list[RetrievedChunk]
    total_candidates: int
    filter_profile_used: str
    empty_filter_warning: bool = False

@dataclass(frozen=True)
class LatencyBreakdown:
    dense_latency_s: float = 0.0
    sparse_latency_s: float = 0.0
    fusion_latency_s: float = 0.0
    cross_encoder_latency_s: float = 0.0
    total_latency_s: float = 0.0

    def to_dict(self) -> dict[str, float]:
        return {
            'dense_latency_s': round(self.dense_latency_s, 4),
            'sparse_latency_s': round(self.sparse_latency_s, 4),
            'fusion_latency_s': round(self.fusion_latency_s, 4),
            'cross_encoder_latency_s': round(self.cross_encoder_latency_s, 4),
            'total_latency_s': round(self.total_latency_s, 4),
        }

print('Schema classes defined.')

In [ ]:
# === EVALUATION METRICS ===
# (Inline from src/evaluation/metrics.py)

_PUNCT_RE = re.compile(r'[^\w\s]', flags=re.UNICODE)
_SPACE_RE = re.compile(r'\s+')

def normalize_text(text: Any) -> str:
    text = '' if text is None else str(text)
    text = unicodedata.normalize('NFC', text).lower()
    text = _PUNCT_RE.sub(' ', text)
    return _SPACE_RE.sub(' ', text).strip()

def tokenize(text: Any) -> list[str]:
    normalized = normalize_text(text)
    return normalized.split() if normalized else []

def recall_at_k(retrieved_ids: list[str], relevant_ids: set[str], k: int) -> float:
    if not relevant_ids:
        return 0.0
    return len(set(retrieved_ids[:k]) & relevant_ids) / len(relevant_ids)

def hit_at_k(retrieved_ids: list[str], relevant_ids: set[str], k: int) -> float:
    return 1.0 if relevant_ids and set(retrieved_ids[:k]) & relevant_ids else 0.0

def mrr_at_k(retrieved_ids: list[str], relevant_ids: set[str], k: int) -> float:
    if not relevant_ids:
        return 0.0
    for index, chunk_id in enumerate(retrieved_ids[:k], start=1):
        if chunk_id in relevant_ids:
            return 1.0 / index
    return 0.0

def ndcg_at_k(retrieved_ids: list[str], relevant_ids: set[str], k: int) -> float:
    if not relevant_ids:
        return 0.0
    dcg = 0.0
    for index, chunk_id in enumerate(retrieved_ids[:k], start=1):
        if chunk_id in relevant_ids:
            dcg += 1.0 / math.log2(index + 1)
    ideal_hits = min(len(relevant_ids), k)
    ideal = sum(1.0 / math.log2(i + 1) for i in range(1, ideal_hits + 1))
    return dcg / ideal if ideal else 0.0

def jaccard_at_k(retrieved_ids: list[str], relevant_ids: set[str], k: int) -> float:
    retrieved = set(retrieved_ids[:k])
    union = retrieved | relevant_ids
    if not union:
        return 0.0
    return len(retrieved & relevant_ids) / len(union)

def aggregate_metrics(rows: list[dict[str, Any]], metric_keys: list[str]) -> dict[str, Any]:
    out = {'count': len(rows)}
    for key in metric_keys:
        values = [float(row.get(key, 0.0)) for row in rows]
        out[key] = sum(values) / len(values) if values else 0.0
    return out

def aggregate_by(rows: list[dict[str, Any]], field_name: str, metric_keys: list[str]) -> dict[str, Any]:
    groups: dict[str, list] = defaultdict(list)
    for row in rows:
        groups[str(row.get(field_name) or 'unknown')].append(row)
    return {name: aggregate_metrics(group, metric_keys) for name, group in sorted(groups.items())}

print('Evaluation metrics defined.')

In [ ]:
# === BM25 SPARSE RETRIEVER (inline) ===

def simple_tokenize(text: str) -> list[str]:
    """Unicode-aware whitespace tokenizer with punctuation removal."""
    text = unicodedata.normalize('NFC', text).lower()
    text = re.sub(r'[^\w\s]', ' ', text, flags=re.UNICODE)
    return [tok for tok in text.split() if len(tok) > 1]

def get_tokenizer():
    """Return the best available Vietnamese tokenizer."""
    try:
        from underthesea import word_tokenize
        def _tokenize(text: str) -> list[str]:
            segmented = word_tokenize(text, format='text')
            return simple_tokenize(segmented)
        print('Using underthesea word_tokenize for BM25 tokenization')
        return _tokenize
    except ImportError:
        print('underthesea not available; using simple whitespace tokenizer')
        return simple_tokenize

class BM25SparseRetriever:
    """BM25Okapi-based sparse retriever."""

    def __init__(self, *, bm25, chunk_ids, payloads, tokenizer):
        self._bm25 = bm25
        self._chunk_ids = chunk_ids
        self._payloads = payloads
        self._tokenizer = tokenizer

    @classmethod
    def build_from_records(cls, records, *, text_field='chunk_text', id_field='chunk_id', tokenizer=None):
        from rank_bm25 import BM25Okapi
        if tokenizer is None:
            tokenizer = get_tokenizer()
        chunk_ids, payloads, tokenized = [], [], []
        for record in records:
            chunk_ids.append(str(record.get(id_field) or ''))
            payloads.append(record)
            tokenized.append(tokenizer(str(record.get(text_field) or '')))
        bm25 = BM25Okapi(tokenized)
        return cls(bm25=bm25, chunk_ids=chunk_ids, payloads=payloads, tokenizer=tokenizer)

    def search(self, query: str, *, top_k: int = 20) -> list[SearchHit]:
        tokenized_query = self._tokenizer(query)
        scores = self._bm25.get_scores(tokenized_query)
        top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]
        hits = []
        for idx in top_indices:
            score = float(scores[idx])
            if score <= 0:
                continue
            hits.append(SearchHit(
                point_id=self._chunk_ids[idx],
                score=score,
                payload=self._payloads[idx],
            ))
        return hits

    def search_with_latency(self, query: str, *, top_k: int = 20):
        t0 = time.perf_counter()
        hits = self.search(query, top_k=top_k)
        return hits, time.perf_counter() - t0

    def save(self, index_dir: Path):
        index_dir.mkdir(parents=True, exist_ok=True)
        with (index_dir / 'bm25_index.pkl').open('wb') as f:
            pickle.dump(self._bm25, f, protocol=pickle.HIGHEST_PROTOCOL)
        with (index_dir / 'bm25_metadata.pkl').open('wb') as f:
            pickle.dump({'chunk_ids': self._chunk_ids, 'payloads': self._payloads}, f, protocol=pickle.HIGHEST_PROTOCOL)
        print(f'Saved BM25 index to {index_dir} ({len(self._chunk_ids):,} documents)')

    @classmethod
    def load(cls, index_dir: Path):
        with (index_dir / 'bm25_index.pkl').open('rb') as f:
            bm25 = pickle.load(f)
        with (index_dir / 'bm25_metadata.pkl').open('rb') as f:
            meta = pickle.load(f)
        tokenizer = get_tokenizer()
        print(f'Loaded BM25 index: {len(meta["chunk_ids"]):,} documents')
        return cls(bm25=bm25, chunk_ids=meta['chunk_ids'], payloads=meta['payloads'], tokenizer=tokenizer)

    @property
    def total_documents(self):
        return len(self._chunk_ids)

print('BM25SparseRetriever defined.')

In [ ]:
# === DENSE RETRIEVER WRAPPER ===
# Wraps FAISS index + SentenceTransformer embedder for Dense search

class DenseRetriever:
    """Minimal Dense retriever using FAISS index + embedder."""

    def __init__(self, *, embedder, faiss_index, payloads, id_map, query_prefix='query: '):
        self._embedder = embedder
        self._index = faiss_index
        self._payloads = payloads    # dict[int, dict]
        self._id_map = id_map        # dict[int, str]  int_id -> chunk_id
        self._query_prefix = query_prefix

    def search(self, query: str, *, top_k: int = 30, score_threshold: float = 0.0) -> list[SearchHit]:
        """Embed query and search FAISS index."""
        prefixed = self._query_prefix + query
        vectors = self._embedder.encode([prefixed], normalize_embeddings=True)
        query_vector = np.array(vectors, dtype=np.float32)

        search_limit = min(top_k * 3, self._index.ntotal)
        scores, indices = self._index.search(query_vector, search_limit)

        hits = []
        for score, idx in zip(scores[0], indices[0]):
            if idx < 0:
                continue
            if float(score) < score_threshold:
                continue
            payload = self._payloads.get(int(idx), {})
            point_id = self._id_map.get(int(idx), str(idx))
            hits.append(SearchHit(point_id=point_id, score=float(score), payload=payload))
            if len(hits) >= top_k:
                break
        return hits

    def search_with_latency(self, query: str, *, top_k: int = 30, score_threshold: float = 0.0):
        t0 = time.perf_counter()
        hits = self.search(query, top_k=top_k, score_threshold=score_threshold)
        return hits, time.perf_counter() - t0

print('DenseRetriever defined.')

In [ ]:
# === HYBRID RETRIEVER with RRF + Cross-Encoder ===

class HybridRetriever:
    """
    Hybrid Dense + Sparse retriever.
    
    Configs:
    - use_rrf=False, use_cross_encoder=False → simple score merge + dedup (Rerank-None-Hybrid)
    - use_rrf=True,  use_cross_encoder=False → RRF fusion (Rerank-RRF-Hybrid)
    - use_rrf=False, use_cross_encoder=True  → merge + CE rerank (Rerank-CrossEncoder-Hybrid)
    - use_rrf=True,  use_cross_encoder=True  → RRF + CE (Rerank-RRFPlusCrossEncoder-Hybrid)
    """

    def __init__(self, *, dense, sparse, cross_encoder=None, use_rrf=False, use_cross_encoder=False, rrf_k=60):
        self.dense = dense
        self.sparse = sparse
        self.use_rrf = use_rrf
        self.use_cross_encoder = use_cross_encoder
        self.rrf_k = rrf_k
        self._cross_encoder = cross_encoder

    @staticmethod
    def merge_by_score(dense_hits, sparse_hits):
        """Simple score-based merge with normalization + dedup by chunk_id."""
        dense_max = max((h.score for h in dense_hits), default=1.0) or 1.0
        sparse_max = max((h.score for h in sparse_hits), default=1.0) or 1.0
        combined = {}
        for hit in dense_hits:
            cid = str(hit.payload.get('chunk_id') or hit.point_id)
            norm = hit.score / dense_max
            if cid not in combined or norm > combined[cid].score:
                combined[cid] = SearchHit(point_id=hit.point_id, score=norm, payload=hit.payload)
        for hit in sparse_hits:
            cid = str(hit.payload.get('chunk_id') or hit.point_id)
            norm = hit.score / sparse_max
            if cid not in combined or norm > combined[cid].score:
                combined[cid] = SearchHit(point_id=hit.point_id, score=norm, payload=hit.payload)
        return sorted(combined.values(), key=lambda h: h.score, reverse=True)

    @staticmethod
    def rrf_fusion(dense_hits, sparse_hits, *, k=60):
        """Reciprocal Rank Fusion: score(d) = Σ 1/(k + rank_i(d))"""
        rrf_scores = {}
        best_hit = {}
        for rank, hit in enumerate(dense_hits, start=1):
            cid = str(hit.payload.get('chunk_id') or hit.point_id)
            rrf_scores[cid] = rrf_scores.get(cid, 0.0) + 1.0 / (k + rank)
            if cid not in best_hit:
                best_hit[cid] = hit
        for rank, hit in enumerate(sparse_hits, start=1):
            cid = str(hit.payload.get('chunk_id') or hit.point_id)
            rrf_scores[cid] = rrf_scores.get(cid, 0.0) + 1.0 / (k + rank)
            if cid not in best_hit:
                best_hit[cid] = hit
        sorted_ids = sorted(rrf_scores.keys(), key=lambda c: rrf_scores[c], reverse=True)
        return [
            SearchHit(point_id=best_hit[cid].point_id, score=rrf_scores[cid], payload=best_hit[cid].payload)
            for cid in sorted_ids
        ]

    def cross_encoder_rerank(self, query, hits, *, top_n):
        """Rerank using Cross-Encoder. Returns (reranked_hits, latency_s)."""
        if self._cross_encoder is None or not hits:
            return hits[:top_n], 0.0
        t0 = time.perf_counter()
        pairs = [(query, str(h.payload.get('chunk_text') or '')) for h in hits]
        ce_scores = self._cross_encoder.predict(pairs)
        scored = [
            SearchHit(point_id=h.point_id, score=float(s), payload=h.payload)
            for h, s in zip(hits, ce_scores)
        ]
        scored.sort(key=lambda h: h.score, reverse=True)
        return scored[:top_n], time.perf_counter() - t0

    def retrieve_with_latency(self, query, *, top_k=30, top_n=10, score_threshold=0.0):
        """Full hybrid pipeline → (hits, LatencyBreakdown)"""
        total_t0 = time.perf_counter()

        # Dense
        dense_hits, dense_lat = self.dense.search_with_latency(query, top_k=top_k, score_threshold=score_threshold)
        # Sparse
        sparse_hits, sparse_lat = self.sparse.search_with_latency(query, top_k=top_k)

        # Fusion
        fusion_t0 = time.perf_counter()
        if self.use_rrf:
            fused = self.rrf_fusion(dense_hits, sparse_hits, k=self.rrf_k)
        else:
            fused = self.merge_by_score(dense_hits, sparse_hits)
        fusion_lat = time.perf_counter() - fusion_t0

        # Cross-Encoder
        ce_lat = 0.0
        if self.use_cross_encoder and self._cross_encoder is not None:
            candidates = fused[:top_n * 3]
            fused, ce_lat = self.cross_encoder_rerank(query, candidates, top_n=top_n)
        else:
            fused = fused[:top_n]

        total_lat = time.perf_counter() - total_t0
        latency = LatencyBreakdown(
            dense_latency_s=dense_lat,
            sparse_latency_s=sparse_lat,
            fusion_latency_s=fusion_lat,
            cross_encoder_latency_s=ce_lat,
            total_latency_s=total_lat,
        )
        return fused, latency

print('HybridRetriever defined.')

## 4. Load Data & Build Indices

In [ ]:
# === LOAD CHUNKS ===
print(f'Loading chunks from {CHUNKS_PATH} ...')
chunks = []
with open(CHUNKS_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            chunks.append(json.loads(line))
print(f'Loaded {len(chunks):,} chunks')

# Show sample
if chunks:
    sample = chunks[0]
    print(f'Sample fields: {list(sample.keys())[:15]}')
    print(f'Sample chunk_id: {sample.get("chunk_id")}')
    print(f'Sample text[:200]: {str(sample.get("chunk_text") or "")[:200]}')

In [ ]:
# === LOAD BENCHMARK QA ===
print(f'Loading benchmark from {BENCHMARK_PATH} ...')
qa_data = []
with open(BENCHMARK_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            qa_data.append(json.loads(line))
print(f'Loaded {len(qa_data):,} QA items')

# Filter: only answerable QA with ground truth chunk_ids
eval_qa = []
skipped_unanswerable = 0
skipped_no_gt = 0
for qa in qa_data:
    answer_type = str(qa.get('answer_type') or '').lower()
    category = str(qa.get('category') or '').lower()
    if answer_type == 'unanswerable' or category == 'unanswerable':
        skipped_unanswerable += 1
        continue
    gt = qa.get('ground_truth') or {}
    gt_chunks = {str(cid) for cid in gt.get('chunk_ids') or [] if cid}
    if not gt_chunks:
        skipped_no_gt += 1
        continue
    eval_qa.append((qa, gt_chunks))

if ABLATION_LIMIT:
    eval_qa = eval_qa[:ABLATION_LIMIT]

print(f'Evaluable QA: {len(eval_qa)}')
print(f'Skipped unanswerable: {skipped_unanswerable}')
print(f'Skipped no ground truth: {skipped_no_gt}')

In [ ]:
# === LOAD FAISS INDEX (Dense) ===
import faiss
from sentence_transformers import SentenceTransformer

print(f'Loading FAISS index from {INDEX_DIR} ...')
faiss_index = faiss.read_index(str(INDEX_DIR / 'index.faiss'))
print(f'FAISS index: {faiss_index.ntotal:,} vectors, dim={faiss_index.d}')

# Load payloads
print('Loading payloads...')
dense_payloads = {}
payloads_path = INDEX_DIR / 'payloads.jsonl'
with open(payloads_path, 'r', encoding='utf-8') as f:
    for line_no, line in enumerate(f):
        line = line.strip()
        if line:
            dense_payloads[line_no] = json.loads(line)
print(f'Loaded {len(dense_payloads):,} payloads')

# Load ID map
id_map_path = INDEX_DIR / 'id_map.json'
dense_id_map = {}  # int -> str
if id_map_path.exists():
    with open(id_map_path, 'r', encoding='utf-8') as f:
        raw = json.load(f)
    dense_id_map = {int(v): str(k) for k, v in raw.items()}
    print(f'Loaded ID map: {len(dense_id_map):,} entries')
else:
    # Fallback: use sequential IDs
    dense_id_map = {i: str(dense_payloads.get(i, {}).get('chunk_id', i)) for i in dense_payloads}
    print(f'No id_map.json; using payload chunk_ids')

# Load embedder
print(f'Loading embedding model: {EMBEDDING_MODEL} ...')
embedder = SentenceTransformer(EMBEDDING_MODEL)
print(f'Embedder loaded. Dimension: {embedder.get_sentence_embedding_dimension()}')

# Build Dense retriever
dense_retriever = DenseRetriever(
    embedder=embedder,
    faiss_index=faiss_index,
    payloads=dense_payloads,
    id_map=dense_id_map,
    query_prefix='query: ',
)
print('Dense retriever ready.')

In [ ]:
# === BUILD BM25 SPARSE INDEX ===

# Check if sparse index already exists
if (SPARSE_INDEX_DIR / 'bm25_index.pkl').exists():
    print('Loading existing BM25 sparse index...')
    sparse_retriever = BM25SparseRetriever.load(SPARSE_INDEX_DIR)
else:
    print('Building BM25 sparse index from chunks...')
    t0 = time.perf_counter()
    sparse_retriever = BM25SparseRetriever.build_from_records(chunks)
    build_time = time.perf_counter() - t0
    print(f'BM25 index built: {sparse_retriever.total_documents:,} documents in {build_time:.2f}s')
    
    # Save for reuse
    sparse_retriever.save(SPARSE_INDEX_DIR)

# Smoke test
print('\n--- Sparse smoke test ---')
test_q = 'Điều kiện để người lao động đơn phương chấm dứt hợp đồng lao động'
test_hits, test_lat = sparse_retriever.search_with_latency(test_q, top_k=5)
print(f'Query: {test_q}')
print(f'Latency: {test_lat:.4f}s')
print(f'Results: {len(test_hits)}')
for rank, hit in enumerate(test_hits, start=1):
    print(f'  [{rank}] score={hit.score:.4f} chunk_id={hit.payload.get("chunk_id", hit.point_id)}')

In [ ]:
# === LOAD CROSS-ENCODER (optional — may be slow) ===
from sentence_transformers import CrossEncoder

print(f'Loading Cross-Encoder: {CROSS_ENCODER_MODEL} ...')
ce_t0 = time.perf_counter()
cross_encoder = CrossEncoder(CROSS_ENCODER_MODEL)
print(f'Cross-Encoder loaded in {time.perf_counter() - ce_t0:.2f}s')

# Quick CE smoke test
test_pairs = [(test_q, str(test_hits[0].payload.get('chunk_text', ''))[:500])]
ce_test_score = cross_encoder.predict([test_pairs[0]])
print(f'CE smoke test score: {ce_test_score[0]:.4f}')

## 5. Define Ablation Configs

In [ ]:
# === ABLATION CONFIG MATRIX ===
# 5 configs as specified by Người 1 (Bình)

ABLATION_CONFIGS = {
    'Retrieval-Hybrid-SparseDense': {
        'description': 'So sánh Hybrid với Dense — simple score merge + dedup, không RRF, không CE',
        'use_rrf': False,
        'use_cross_encoder': False,
    },
    'Rerank-None-Hybrid': {
        'description': 'Hybrid baseline — không rerank (không RRF, không CE), chỉ dedup + merge score',
        'use_rrf': False,
        'use_cross_encoder': False,
    },
    'Rerank-RRF-Hybrid': {
        'description': 'Hybrid dùng RRF fusion (k=60), không CE',
        'use_rrf': True,
        'use_cross_encoder': False,
    },
    'Rerank-CrossEncoder-Hybrid': {
        'description': 'Hybrid dùng CE rerank (không RRF), merge + CE',
        'use_rrf': False,
        'use_cross_encoder': True,
    },
    'Rerank-RRFPlusCrossEncoder-Hybrid': {
        'description': 'RRF lấy candidates + CE rerank trên top candidates',
        'use_rrf': True,
        'use_cross_encoder': True,
    },
}

print(f'Defined {len(ABLATION_CONFIGS)} ablation configs:')
for name, cfg in ABLATION_CONFIGS.items():
    print(f'  {name}: rrf={cfg["use_rrf"]}, ce={cfg["use_cross_encoder"]} — {cfg["description"]}')

## 6. Run Full Ablation

In [ ]:
# === ABLATION RUNNER ===

METRIC_NAMES = ['recall', 'hit', 'mrr', 'ndcg', 'jaccard']
METRIC_KEYS = [f'{name}@{k}' for k in TOP_K_EVAL for name in METRIC_NAMES]

def run_single_ablation(config_name, config, eval_qa, dense_retriever, sparse_retriever, cross_encoder):
    """Run one ablation config over all QA items. Returns (cases, latencies, summary)."""
    print(f'\n{"="*80}')
    print(f'Running: {config_name}')
    print(f'Description: {config["description"]}')
    print(f'use_rrf={config["use_rrf"]}, use_cross_encoder={config["use_cross_encoder"]}')
    print(f'Evaluating {len(eval_qa)} QA items...')
    print(f'{"="*80}')

    # Build hybrid retriever for this config
    ce = cross_encoder if config['use_cross_encoder'] else None
    hybrid = HybridRetriever(
        dense=dense_retriever,
        sparse=sparse_retriever,
        cross_encoder=ce,
        use_rrf=config['use_rrf'],
        use_cross_encoder=config['use_cross_encoder'],
        rrf_k=RRF_K,
    )

    cases = []
    latencies = []
    run_t0 = time.perf_counter()

    for idx, (qa, gt_chunks) in enumerate(eval_qa):
        question = str(qa.get('question') or '')
        qa_id = str(qa.get('qa_id') or qa.get('id') or f'qa_{idx+1}')

        try:
            hits, latency = hybrid.retrieve_with_latency(
                question,
                top_k=TOP_K,
                top_n=TOP_N,
                score_threshold=SCORE_THRESHOLD,
            )
            retrieved_ids = [str(h.payload.get('chunk_id') or h.point_id) for h in hits]

            # Compute metrics
            row = {
                'qa_id': qa_id,
                'question': question,
                'category': qa.get('category'),
                'difficulty': qa.get('difficulty'),
                'answer_type': qa.get('answer_type'),
                'ground_truth_chunk_ids': sorted(gt_chunks),
                'retrieved_chunk_ids': retrieved_ids,
                'retrieved': [
                    {
                        'rank': rank,
                        'chunk_id': str(h.payload.get('chunk_id') or h.point_id),
                        'score': round(h.score, 6),
                        'citation': str(h.payload.get('citation_anchor') or h.payload.get('citation_label') or ''),
                    }
                    for rank, h in enumerate(hits, start=1)
                ],
            }

            for k in TOP_K_EVAL:
                row[f'recall@{k}'] = recall_at_k(retrieved_ids, gt_chunks, k)
                row[f'hit@{k}'] = hit_at_k(retrieved_ids, gt_chunks, k)
                row[f'mrr@{k}'] = mrr_at_k(retrieved_ids, gt_chunks, k)
                row[f'ndcg@{k}'] = ndcg_at_k(retrieved_ids, gt_chunks, k)
                row[f'jaccard@{k}'] = jaccard_at_k(retrieved_ids, gt_chunks, k)

            cases.append(row)
            latencies.append(latency.to_dict())

        except Exception as e:
            print(f'  ERROR at qa_id={qa_id}: {e}')
            cases.append({'qa_id': qa_id, 'error': str(e)})
            latencies.append(LatencyBreakdown().to_dict())

        if (idx + 1) % 25 == 0:
            print(f'  Evaluated {idx+1}/{len(eval_qa)} ...')

    run_duration = time.perf_counter() - run_t0
    print(f'  Completed {len(cases)} cases in {run_duration:.2f}s')

    # Aggregate metrics
    valid_cases = [c for c in cases if 'error' not in c]
    summary = {
        'config_name': config_name,
        'config': config,
        'counts': {
            'total': len(cases),
            'evaluated': len(valid_cases),
            'errors': len(cases) - len(valid_cases),
        },
        'overall': aggregate_metrics(valid_cases, METRIC_KEYS),
        'by_category': aggregate_by(valid_cases, 'category', METRIC_KEYS),
        'by_difficulty': aggregate_by(valid_cases, 'difficulty', METRIC_KEYS),
        'by_answer_type': aggregate_by(valid_cases, 'answer_type', METRIC_KEYS),
        'latency': {
            'total_run_time_s': round(run_duration, 2),
            'avg': {
                k: round(np.mean([lat[k] for lat in latencies]), 4)
                for k in latencies[0].keys()
            } if latencies else {},
            'median': {
                k: round(float(np.median([lat[k] for lat in latencies])), 4)
                for k in latencies[0].keys()
            } if latencies else {},
        },
    }

    # Print summary
    print(f'\n--- {config_name} Results ---')
    for key in METRIC_KEYS:
        print(f'  {key}: {summary["overall"].get(key, 0):.4f}')
    print(f'  Avg latency: {summary["latency"]["avg"].get("total_latency_s", 0):.4f}s')

    return cases, latencies, summary

print('Ablation runner defined.')

In [ ]:
# === RUN ALL ABLATION CONFIGS ===

all_results = {}  # config_name -> (cases, latencies, summary)

for config_name, config in ABLATION_CONFIGS.items():
    cases, latencies, summary = run_single_ablation(
        config_name, config, eval_qa,
        dense_retriever, sparse_retriever, cross_encoder,
    )
    all_results[config_name] = (cases, latencies, summary)

print(f'\n{"="*80}')
print(f'All {len(all_results)} ablation configs completed!')
print(f'{"="*80}')

## 7. Save Results

In [ ]:
# === SAVE RESULTS TO evaluation_runs/ablation/<config>/ ===

for config_name, (cases, latencies, summary) in all_results.items():
    run_dir = OUTPUT_BASE / config_name
    run_dir.mkdir(parents=True, exist_ok=True)

    # manifest.json
    manifest = {
        'run_id': config_name,
        'config_name': config_name,
        'benchmark_path': str(BENCHMARK_PATH),
        'benchmark_version': 'qa_final_v1',
        'corpus_version': 'v2',
        'index_path': str(INDEX_DIR),
        'sparse_index_path': str(SPARSE_INDEX_DIR),
        'retriever_config': {
            'type': 'hybrid',
            'embedding_model': EMBEDDING_MODEL,
            'cross_encoder_model': CROSS_ENCODER_MODEL if summary['config']['use_cross_encoder'] else None,
            'use_rrf': summary['config']['use_rrf'],
            'use_cross_encoder': summary['config']['use_cross_encoder'],
            'rrf_k': RRF_K,
            'top_k': TOP_K,
            'top_n': TOP_N,
            'score_threshold': SCORE_THRESHOLD,
            'filter_profile': FILTER_PROFILE,
        },
        'generator_config': None,
        'filter_profile': FILTER_PROFILE,
        'timestamp': datetime.now(timezone.utc).isoformat(),
    }
    with (run_dir / 'manifest.json').open('w', encoding='utf-8') as f:
        json.dump(manifest, f, ensure_ascii=False, indent=2)

    # retrieval_cases.jsonl
    with (run_dir / 'retrieval_cases.jsonl').open('w', encoding='utf-8') as f:
        for case in cases:
            f.write(json.dumps(case, ensure_ascii=False, separators=(',', ':')) + '\n')

    # retrieval_metrics.json
    with (run_dir / 'retrieval_metrics.json').open('w', encoding='utf-8') as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

    # latency.json
    with (run_dir / 'latency.json').open('w', encoding='utf-8') as f:
        json.dump({
            'config_name': config_name,
            'per_query_latencies': latencies,
            'avg': summary['latency']['avg'],
            'median': summary['latency']['median'],
            'total_run_time_s': summary['latency']['total_run_time_s'],
        }, f, ensure_ascii=False, indent=2)

    # report.md
    report_lines = [
        f'# Retrieval Ablation Report — {config_name}',
        '',
        f'- Config: `{config_name}`',
        f'- Description: {summary["config"]["description"]}',
        f'- use_rrf: {summary["config"]["use_rrf"]}',
        f'- use_cross_encoder: {summary["config"]["use_cross_encoder"]}',
        f'- Evaluated: {summary["counts"]["evaluated"]}',
        f'- Errors: {summary["counts"]["errors"]}',
        '',
        '## Overall Metrics',
        '',
        '| Metric | Value |',
        '| --- | ---: |',
    ]
    for key in METRIC_KEYS:
        report_lines.append(f'| {key} | {summary["overall"].get(key, 0):.4f} |')
    report_lines.extend([
        '',
        '## Latency',
        '',
        '| Stage | Avg (s) | Median (s) |',
        '| --- | ---: | ---: |',
    ])
    for stage in ['dense_latency_s', 'sparse_latency_s', 'fusion_latency_s', 'cross_encoder_latency_s', 'total_latency_s']:
        avg_v = summary['latency']['avg'].get(stage, 0)
        med_v = summary['latency']['median'].get(stage, 0)
        report_lines.append(f'| {stage} | {avg_v:.4f} | {med_v:.4f} |')

    with (run_dir / 'report.md').open('w', encoding='utf-8') as f:
        f.write('\n'.join(report_lines) + '\n')

    print(f'Saved: {run_dir}/')
    print(f'  manifest.json, retrieval_cases.jsonl, retrieval_metrics.json, latency.json, report.md')

print(f'\nAll results saved to {OUTPUT_BASE}/')

## 8. Comparison Table & Analysis

In [ ]:
# === COMPARISON TABLE ===

try:
    import pandas as pd
    from IPython.display import display, HTML
    HAS_PANDAS = True
except ImportError:
    HAS_PANDAS = False

# Build comparison dataframe
comparison_rows = []
for config_name, (_, _, summary) in all_results.items():
    row = {'Config': config_name}
    row['RRF'] = '✅' if summary['config']['use_rrf'] else '❌'
    row['CE'] = '✅' if summary['config']['use_cross_encoder'] else '❌'
    for key in METRIC_KEYS:
        row[key] = round(summary['overall'].get(key, 0), 4)
    row['Avg Latency (s)'] = round(summary['latency']['avg'].get('total_latency_s', 0), 4)
    comparison_rows.append(row)

if HAS_PANDAS:
    df_comparison = pd.DataFrame(comparison_rows)
    print('\n=== ABLATION COMPARISON TABLE ===')
    display(df_comparison)
else:
    print('\n=== ABLATION COMPARISON TABLE ===')
    # Fallback: print as text table
    for row in comparison_rows:
        print(json.dumps(row, ensure_ascii=False, indent=2))

In [ ]:
# === LATENCY COMPARISON TABLE ===

latency_rows = []
for config_name, (_, _, summary) in all_results.items():
    row = {'Config': config_name}
    for stage in ['dense_latency_s', 'sparse_latency_s', 'fusion_latency_s', 'cross_encoder_latency_s', 'total_latency_s']:
        row[f'Avg {stage}'] = round(summary['latency']['avg'].get(stage, 0), 4)
        row[f'Median {stage}'] = round(summary['latency']['median'].get(stage, 0), 4)
    latency_rows.append(row)

if HAS_PANDAS:
    df_latency = pd.DataFrame(latency_rows)
    print('\n=== LATENCY COMPARISON ===')
    display(df_latency)
else:
    print('\n=== LATENCY COMPARISON ===')
    for row in latency_rows:
        print(json.dumps(row, ensure_ascii=False, indent=2))

In [ ]:
# === CHARTS (optional — if matplotlib available) ===

try:
    import matplotlib.pyplot as plt

    config_names = list(all_results.keys())
    short_names = [
        'Hybrid\n(no rerank)',
        'None\nHybrid',
        'RRF\nHybrid',
        'CE\nHybrid',
        'RRF+CE\nHybrid',
    ]

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    # Chart 1: Recall@k
    for k in TOP_K_EVAL:
        values = [all_results[c][2]['overall'].get(f'recall@{k}', 0) for c in config_names]
        axes[0].bar(
            [x + TOP_K_EVAL.index(k) * 0.15 for x in range(len(config_names))],
            values, width=0.15, label=f'Recall@{k}'
        )
    axes[0].set_xticks(range(len(config_names)))
    axes[0].set_xticklabels(short_names, fontsize=8)
    axes[0].set_title('Recall@k by Config')
    axes[0].legend(fontsize=8)
    axes[0].set_ylim(0, 1)

    # Chart 2: MRR & nDCG @10
    mrr_vals = [all_results[c][2]['overall'].get('mrr@10', 0) for c in config_names]
    ndcg_vals = [all_results[c][2]['overall'].get('ndcg@10', 0) for c in config_names]
    x = range(len(config_names))
    axes[1].bar([i - 0.15 for i in x], mrr_vals, width=0.3, label='MRR@10', color='steelblue')
    axes[1].bar([i + 0.15 for i in x], ndcg_vals, width=0.3, label='nDCG@10', color='coral')
    axes[1].set_xticks(range(len(config_names)))
    axes[1].set_xticklabels(short_names, fontsize=8)
    axes[1].set_title('MRR@10 & nDCG@10')
    axes[1].legend()
    axes[1].set_ylim(0, 1)

    # Chart 3: Latency breakdown
    stages = ['dense_latency_s', 'sparse_latency_s', 'fusion_latency_s', 'cross_encoder_latency_s']
    stage_labels = ['Dense', 'Sparse', 'Fusion', 'CE']
    bottom = np.zeros(len(config_names))
    colors = ['#4e79a7', '#f28e2b', '#e15759', '#76b7b2']
    for stage, label, color in zip(stages, stage_labels, colors):
        values = np.array([all_results[c][2]['latency']['avg'].get(stage, 0) for c in config_names])
        axes[2].bar(range(len(config_names)), values, bottom=bottom, label=label, color=color)
        bottom += values
    axes[2].set_xticks(range(len(config_names)))
    axes[2].set_xticklabels(short_names, fontsize=8)
    axes[2].set_title('Avg Latency Breakdown (s)')
    axes[2].legend(fontsize=8)

    plt.tight_layout()
    plt.savefig(str(OUTPUT_BASE / 'ablation_comparison_charts.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Charts saved to {OUTPUT_BASE / "ablation_comparison_charts.png"}')

except ImportError:
    print('matplotlib not available; skipping charts.')

## 9. Analysis & Conclusions

### Nhận xét Hybrid/Reranker cho report

In [ ]:
# === AUTO-GENERATE ANALYSIS NOTES ===

print('='*80)
print('NHẬN XÉT HYBRID/RERANKER — Người 3 (Kiệt)')
print('='*80)

# Get summaries
s_hybrid = all_results['Retrieval-Hybrid-SparseDense'][2]
s_none = all_results['Rerank-None-Hybrid'][2]
s_rrf = all_results['Rerank-RRF-Hybrid'][2]
s_ce = all_results['Rerank-CrossEncoder-Hybrid'][2]
s_rrf_ce = all_results['Rerank-RRFPlusCrossEncoder-Hybrid'][2]

print('\n1. Hybrid (Sparse+Dense) so với Dense-only baseline:')
print(f'   Hybrid Recall@10: {s_hybrid["overall"].get("recall@10", 0):.4f}')
print(f'   (So sánh với Dense-only từ Người 2 để kết luận)')

print('\n2. RRF có tốt hơn raw hybrid không:')
r10_none = s_none['overall'].get('recall@10', 0)
r10_rrf = s_rrf['overall'].get('recall@10', 0)
diff_rrf = r10_rrf - r10_none
print(f'   No-rerank Recall@10: {r10_none:.4f}')
print(f'   RRF Recall@10:       {r10_rrf:.4f}')
print(f'   Difference:          {diff_rrf:+.4f} ({"RRF tốt hơn" if diff_rrf > 0 else "RRF không tốt hơn" if diff_rrf < 0 else "Bằng nhau"})')

print('\n3. Cross-Encoder có cải thiện quality không:')
r10_ce = s_ce['overall'].get('recall@10', 0)
diff_ce = r10_ce - r10_none
mrr_none = s_none['overall'].get('mrr@10', 0)
mrr_ce = s_ce['overall'].get('mrr@10', 0)
print(f'   No-rerank Recall@10: {r10_none:.4f} | MRR@10: {mrr_none:.4f}')
print(f'   CE Recall@10:        {r10_ce:.4f} | MRR@10: {mrr_ce:.4f}')
print(f'   Recall diff:         {diff_ce:+.4f}')
print(f'   MRR diff:            {mrr_ce - mrr_none:+.4f}')

print('\n4. Cross-Encoder latency — quá chậm cho demo/UI không:')
ce_avg_lat = s_ce['latency']['avg'].get('cross_encoder_latency_s', 0)
ce_total_lat = s_ce['latency']['avg'].get('total_latency_s', 0)
none_total_lat = s_none['latency']['avg'].get('total_latency_s', 0)
print(f'   CE avg latency:      {ce_avg_lat:.4f}s')
print(f'   CE total latency:    {ce_total_lat:.4f}s')
print(f'   No-CE total latency: {none_total_lat:.4f}s')
print(f'   Overhead:            {ce_total_lat - none_total_lat:.4f}s')
if ce_total_lat > 5.0:
    print('   ⚠️ CE quá chậm cho real-time UI (>5s)')
elif ce_total_lat > 2.0:
    print('   ⚡ CE chấp nhận được nhưng chậm (2-5s)')
else:
    print('   ✅ CE đủ nhanh cho demo/UI (<2s)')

print('\n5. Config nào đáng chọn làm main pipeline:')
# Find best config by Recall@10
best_config = max(all_results.keys(), key=lambda c: all_results[c][2]['overall'].get('recall@10', 0))
best_r10 = all_results[best_config][2]['overall'].get('recall@10', 0)
best_lat = all_results[best_config][2]['latency']['avg'].get('total_latency_s', 0)
print(f'   Best Recall@10: {best_config} ({best_r10:.4f}, latency={best_lat:.4f}s)')

# Find best quality-latency tradeoff
for c in all_results:
    r10 = all_results[c][2]['overall'].get('recall@10', 0)
    lat = all_results[c][2]['latency']['avg'].get('total_latency_s', 0)
    print(f'   {c}: Recall@10={r10:.4f}, Latency={lat:.4f}s')

print('\n' + '='*80)

In [ ]:
# === SAVE COMBINED SUMMARY ===

combined_summary = {
    'ablation_type': 'hybrid_retrieval_reranker',
    'owner': 'Người 3 (Kiệt)',
    'timestamp': datetime.now(timezone.utc).isoformat(),
    'settings': {
        'embedding_model': EMBEDDING_MODEL,
        'cross_encoder_model': CROSS_ENCODER_MODEL,
        'top_k': TOP_K,
        'top_n': TOP_N,
        'rrf_k': RRF_K,
        'score_threshold': SCORE_THRESHOLD,
        'filter_profile': FILTER_PROFILE,
        'benchmark_path': str(BENCHMARK_PATH),
        'evaluated_qa_count': len(eval_qa),
    },
    'configs': {},
}

for config_name, (_, _, summary) in all_results.items():
    combined_summary['configs'][config_name] = {
        'overall': summary['overall'],
        'latency_avg': summary['latency']['avg'],
        'latency_median': summary['latency']['median'],
    }

with (OUTPUT_BASE / 'hybrid_ablation_summary.json').open('w', encoding='utf-8') as f:
    json.dump(combined_summary, f, ensure_ascii=False, indent=2)

print(f'Combined summary saved to {OUTPUT_BASE / "hybrid_ablation_summary.json"}')
print('\nDone! Tất cả kết quả đã lưu. Sẵn sàng đưa vào report.')